### 리트리버
- 벡터스토어 기반 리트리버

In [5]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from dotenv import load_dotenv
from langchain_teddynote import logging

logging.langsmith("CH10-Retriever")
load_dotenv()

loader = TextLoader("./data/appendix-keywords.txt", encoding="utf-8") # 파일을 로드
documents = loader.load() # 문서를 로드

#문자 기반으로 텍스트를 분할하는 CharacterTextSplitter를 생성
text_splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=0) # chunk_size와 chunk_overlap을 설정

# 로드된 문서를 분할
split_docs = text_splitter.split_documents(documents) # 분할된 문서를 split_docs에 저장

embeddings = OpenAIEmbeddings() # OpenAIEmbeddings를 생성
# FAISS 벡터 스토어를 생성하고 분할된 문서를 임베딩하여 벡터 스토어에 추가

# 분할된 텍스트와 임베딩을 사용하여 FAISS 벡터 데이터베이스를 생성
db = FAISS.from_documents(split_docs, embeddings) # FAISS 벡터 스토어를 생성하고 분할된 문서를 임베딩하여 벡터 스토어에 추가

C:\Users\user\AppData\Local\Temp\ipykernel_10680\2210478385.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


LangSmith 추적을 시작합니다.
[프로젝트명]
CH10-Retriever


In [7]:
retriever = db.as_retriever() # FAISS 벡터 스토어를 검색 가능한 retriever로 변환

In [8]:
docs = retriever.invoke("임베딩(Embedding)은 무엇인가요?") # 관련 문서를 검색

for doc in docs: # 검색된 문서를 출력
    print(doc.page_content)
    print("--------------------------------------------------")

정의: 임베딩은 단어나 문장 같은 텍스트 데이터를 저차원의 연속적인 벡터로 변환하는 과정입니다. 이를 통해 컴퓨터가 텍스트를 이해하고 처리할 수 있게 합니다.
예시: "사과"라는 단어를 [0.65, -0.23, 0.17]과 같은 벡터로 표현합니다.
연관키워드: 자연어 처리, 벡터화, 딥러닝

Token
--------------------------------------------------
정의: Word2Vec은 단어를 벡터 공간에 매핑하여 단어 간의 의미적 관계를 나타내는 자연어 처리 기술입니다. 이는 단어의 문맥적 유사성을 기반으로 벡터를 생성합니다.
예시: Word2Vec 모델에서 "왕"과 "여왕"은 서로 가까운 위치에 벡터로 표현됩니다.
연관키워드: 자연어 처리, 임베딩, 의미론적 유사성
LLM (Large Language Model)
--------------------------------------------------
Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.
예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.
연관키워드: 자연어 처리, 검색 알고리즘, 데이터 마이닝

Embedding
--------------------------------------------------
정의: 크롤링은 자동화된 방식으로 웹 페이지를 방문하여 데이터를 수집하는 과정입니다. 이는 검색 엔진 최적화나 데이터 분석에 자주 사용됩니다.
예시: 구글 검색 엔진이 인터넷 상의 웹사이트를 방문하여 콘텐츠를 수집하고 인덱싱하는 것이 크롤링입니다.
연관키워드: 데이터 수집, 웹 스크래핑, 검색 엔진

Word2Vec
--------------------------------------------------


In [9]:
retriever = db.as_retriever( # FAISS 벡터 스토어를 검색 가능한 retriever로 변환, MMR 검색 유형을 지정
    search_type="mmr", search_kwargs={"k": 2, "fetch_k": 10, "lambda_mult": 0.6}
)
docs = retriever.invoke("임베딩(Embedding)은 무엇인가요?") # 관련 문서를 검색

for doc in docs: # 검색된 문서를 출력
    print(doc.page_content)
    print("--------------------------------------------------")

정의: 임베딩은 단어나 문장 같은 텍스트 데이터를 저차원의 연속적인 벡터로 변환하는 과정입니다. 이를 통해 컴퓨터가 텍스트를 이해하고 처리할 수 있게 합니다.
예시: "사과"라는 단어를 [0.65, -0.23, 0.17]과 같은 벡터로 표현합니다.
연관키워드: 자연어 처리, 벡터화, 딥러닝

Token
--------------------------------------------------
Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.
예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.
연관키워드: 자연어 처리, 검색 알고리즘, 데이터 마이닝

Embedding
--------------------------------------------------


In [12]:
retriever = db.as_retriever(
    search_type="similarity_score_threshold", # 검색 유형 지정
    search_kwargs={"score_threshold":0.8}, # 임계값 지정
)

for doc in retriever.invoke("Word2Vec은 무엇인가요?"): # 관련 문서를 검색
    print(doc.page_content) # 검색된 문서의 내용
    print("--------------------------------------------------")

정의: Word2Vec은 단어를 벡터 공간에 매핑하여 단어 간의 의미적 관계를 나타내는 자연어 처리 기술입니다. 이는 단어의 문맥적 유사성을 기반으로 벡터를 생성합니다.
예시: Word2Vec 모델에서 "왕"과 "여왕"은 서로 가까운 위치에 벡터로 표현됩니다.
연관키워드: 자연어 처리, 임베딩, 의미론적 유사성
LLM (Large Language Model)
--------------------------------------------------


In [13]:
retriever = db.as_retriever(search_kwargs={"k": 1}) # k 값 설정

docs = retriever.invoke("임베딩(Embedding)은 무엇인가요?") # 관련 문서를 검색

for doc in docs: # 관련 문서를 검색
    print(doc.page_content) # 검색된 문서의 내용
    print("--------------------------------------------------")

정의: 임베딩은 단어나 문장 같은 텍스트 데이터를 저차원의 연속적인 벡터로 변환하는 과정입니다. 이를 통해 컴퓨터가 텍스트를 이해하고 처리할 수 있게 합니다.
예시: "사과"라는 단어를 [0.65, -0.23, 0.17]과 같은 벡터로 표현합니다.
연관키워드: 자연어 처리, 벡터화, 딥러닝

Token
--------------------------------------------------


In [14]:
from langchain_core.runnables import ConfigurableField

retriever = db.as_retriever(search_kwargs={"k":1}).configurable_fields( # ConfigurableField("k", int, "검색할 문서 수", default=1) # k 값을 설정할 수 있는 ConfigurableField를 추가
    search_type=ConfigurableField(
        id="search_type",
        name="Search Type",
        description="The search type to use",
    ),
    search_kwargs=ConfigurableField(
        id="search_kwargs",
        name="Search Kwargs",
        description="The search kwargs to use",
    ),
)


In [15]:
#FAISS 검색에서 가장 유사한 문서 3개를 반환
config = {"configurable": {"search_kwargs": {"k":3}}} # k 값을 3으로 설정

# 관련 문서를 검색
docs = retriever.invoke("임베딩(Embedding)은 무엇인가요?", config=config)

for doc in docs: # 검색된 문서를 출력
    print(doc.page_content)
    print("--------------------------------------------------")

정의: 임베딩은 단어나 문장 같은 텍스트 데이터를 저차원의 연속적인 벡터로 변환하는 과정입니다. 이를 통해 컴퓨터가 텍스트를 이해하고 처리할 수 있게 합니다.
예시: "사과"라는 단어를 [0.65, -0.23, 0.17]과 같은 벡터로 표현합니다.
연관키워드: 자연어 처리, 벡터화, 딥러닝

Token
--------------------------------------------------
정의: Word2Vec은 단어를 벡터 공간에 매핑하여 단어 간의 의미적 관계를 나타내는 자연어 처리 기술입니다. 이는 단어의 문맥적 유사성을 기반으로 벡터를 생성합니다.
예시: Word2Vec 모델에서 "왕"과 "여왕"은 서로 가까운 위치에 벡터로 표현됩니다.
연관키워드: 자연어 처리, 임베딩, 의미론적 유사성
LLM (Large Language Model)
--------------------------------------------------
Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.
예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.
연관키워드: 자연어 처리, 검색 알고리즘, 데이터 마이닝

Embedding
--------------------------------------------------


In [19]:
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_upstage import UpstageEmbeddings

loader = TextLoader("./data/appendix-keywords.txt", encoding="utf-8") # 파일을 로드
documents = loader.load() # 문서를 로드

# 문자 기반으로 텍스트를 분할하는 CharacterTextSplitter를 생성
text_splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=0) # chunk_size와 chunk_overlap을 설정

# 로드된 문서를 분할
split_docs = text_splitter.split_documents(documents) # 분할된 문서를 split_docs에 저장

# Upstage 임베딩을 생성 문서용 모델로 사용
doc_embedder = UpstageEmbeddings(model="solar-embedding-1-large-passage") # 문서용 모델을 사용하여 Upstage 임베딩을 생성

# 분할된 텍스트와 임베딩을 사용하여 FAISS 벡터 데이터베이스를 생성
db = FAISS.from_documents(split_docs, doc_embedder) # FAISS 벡터 스토어를 생성하고 분할된 문서를 임베딩하여 벡터 스토어에 추가

# 쿼리용 Upstage 임베딩을 생성. 쿼리용 모델을 사용
query_embedder = UpstageEmbeddings(model="solar-embedding-1-large-query") # 쿼리용 모델을 사용하여 Upstage 임베딩을 생성

# 쿼리 문장을 벡터로 전환
query_vector = query_embedder.embed_query("임베딩(Embedding)은 무엇인가요?") # 쿼리 문장을 벡터로 전환

# 벡터 유사도 검색을 수행하여 가장 유사한 2개의 문서를 반환
db.similarity_search_by_vector(query_vector, k=2) # 벡터 유사도 검색을 수행하여 가장 유사한 2개의 문서를 반환

[Document(id='ab27e000-0358-4bf4-84cf-4e17ad228501', metadata={'source': './data/appendix-keywords.txt'}, page_content='정의: 임베딩은 단어나 문장 같은 텍스트 데이터를 저차원의 연속적인 벡터로 변환하는 과정입니다. 이를 통해 컴퓨터가 텍스트를 이해하고 처리할 수 있게 합니다.\n예시: "사과"라는 단어를 [0.65, -0.23, 0.17]과 같은 벡터로 표현합니다.\n연관키워드: 자연어 처리, 벡터화, 딥러닝\n\nToken'),
 Document(id='45b0aec7-30b3-45ff-82f2-6f645388eb7c', metadata={'source': './data/appendix-keywords.txt'}, page_content='정의: Word2Vec은 단어를 벡터 공간에 매핑하여 단어 간의 의미적 관계를 나타내는 자연어 처리 기술입니다. 이는 단어의 문맥적 유사성을 기반으로 벡터를 생성합니다.\n예시: Word2Vec 모델에서 "왕"과 "여왕"은 서로 가까운 위치에 벡터로 표현됩니다.\n연관키워드: 자연어 처리, 임베딩, 의미론적 유사성\nLLM (Large Language Model)')]

### 문서 압축기

In [22]:
def pretty_print_docs(docs): # 문서를 보기 좋게 출력하는 도우미 함수
    print(
        f"\n{'-' * 100}\n".join(
            [f"문서 {i+1}:\n{d.page_content}" for i, d in enumerate(docs)]
        )
    )

In [26]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from dotenv import load_dotenv
from langchain_teddynote import logging

load_dotenv()
logging.langsmith("CH10-Retriever")

# appendix-keywords.txt 파일에서 문서 로드
loader = TextLoader("./data/appendix-keywords.txt", encoding="utf-8") # 파일을 로드

# 문서를 청크 크기 300자와 청크 간 중복 0으로 분할
text_splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=0) # chunk_size와 chunk_overlap을 설정
texts = loader.load_and_split(text_splitter) # 문서를 분할

# FAISS 벡터 스토어를 생성하고 리트리버로 변환
retriever = FAISS.from_documents(texts, OpenAIEmbeddings()).as_retriever() # FAISS 벡터 스토어를 생성하고 리트리버로 변환

# 검색어와 관련하여 관련 문서를 검색
docs = retriever.invoke("Semantic Search에 대해서 알려줘.")

pretty_print_docs(docs) # 검색된 문서를 보기 좋게 출력

LangSmith 추적을 시작합니다.
[프로젝트명]
CH10-Retriever
문서 1:
Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.
예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.
연관키워드: 자연어 처리, 검색 알고리즘, 데이터 마이닝

Embedding
----------------------------------------------------------------------------------------------------
문서 2:
정의: 키워드 검색은 사용자가 입력한 키워드를 기반으로 정보를 찾는 과정입니다. 이는 대부분의 검색 엔진과 데이터베이스 시스템에서 기본적인 검색 방식으로 사용됩니다.
예시: 사용자가 "커피숍 서울"이라고 검색하면, 관련된 커피숍 목록을 반환합니다.
연관키워드: 검색 엔진, 데이터 검색, 정보 검색

Page Rank
----------------------------------------------------------------------------------------------------
문서 3:
정의: 크롤링은 자동화된 방식으로 웹 페이지를 방문하여 데이터를 수집하는 과정입니다. 이는 검색 엔진 최적화나 데이터 분석에 자주 사용됩니다.
예시: 구글 검색 엔진이 인터넷 상의 웹사이트를 방문하여 콘텐츠를 수집하고 인덱싱하는 것이 크롤링입니다.
연관키워드: 데이터 수집, 웹 스크래핑, 검색 엔진

Word2Vec
----------------------------------------------------------------------------------------------------
문서 4:
정의: 페이지 랭크는 웹 페이지의 중요도를 평가하는 알고리즘으로, 주로 검색 엔진 결과의 순위를 결정하는 데 사용됩니다. 이는 웹 페이

In [29]:
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_classic.retrievers import ContextualCompressionRetriever

# from langchain_teddynote.document_compressors import LLMChainExtractor  # langchain 1.x와 호환되지 않아 langchain_classic 버전 사용
from langchain_openai import ChatOpenAI

# OpenAI 언어 모델 초기화
llm = ChatOpenAI(temperature=0, model="gpt-4o-mini") # OpenAI 언어 모델 초기화

In [30]:
#LLM을 사용하여 문서 압축기 생성
compressor = LLMChainExtractor.from_llm(llm) # LLM을 사용하여 문서 압축기 생성
compression_retriever = ContextualCompressionRetriever(
    # 문서 압축기와 리트리버를 사용하여 컨텍스트 압축 리트리버 생성
    base_compressor = compressor,
    base_retriever = retriever,
)

In [32]:
pretty_print_docs(retriever.invoke("Semantic Search에 대해서 알려줘"))

문서 1:
Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.
예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.
연관키워드: 자연어 처리, 검색 알고리즘, 데이터 마이닝

Embedding
----------------------------------------------------------------------------------------------------
문서 2:
정의: 키워드 검색은 사용자가 입력한 키워드를 기반으로 정보를 찾는 과정입니다. 이는 대부분의 검색 엔진과 데이터베이스 시스템에서 기본적인 검색 방식으로 사용됩니다.
예시: 사용자가 "커피숍 서울"이라고 검색하면, 관련된 커피숍 목록을 반환합니다.
연관키워드: 검색 엔진, 데이터 검색, 정보 검색

Page Rank
----------------------------------------------------------------------------------------------------
문서 3:
정의: 크롤링은 자동화된 방식으로 웹 페이지를 방문하여 데이터를 수집하는 과정입니다. 이는 검색 엔진 최적화나 데이터 분석에 자주 사용됩니다.
예시: 구글 검색 엔진이 인터넷 상의 웹사이트를 방문하여 콘텐츠를 수집하고 인덱싱하는 것이 크롤링입니다.
연관키워드: 데이터 수집, 웹 스크래핑, 검색 엔진

Word2Vec
----------------------------------------------------------------------------------------------------
문서 4:
정의: 페이지 랭크는 웹 페이지의 중요도를 평가하는 알고리즘으로, 주로 검색 엔진 결과의 순위를 결정하는 데 사용됩니다. 이는 웹 페이지 간의 링크 구조를 분석하여 평가합니다.
예시: 구글 검색 엔진은 페이지 랭크

In [33]:
compressed_docs = (
    compression_retriever.invoke("Semantic Search에 대해서 알려줘")
)
pretty_print_docs(compressed_docs)

문서 1:
Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.
예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.


In [35]:
from langchain_classic.retrievers.document_compressors import LLMChainFilter

_filter = LLMChainFilter.from_llm(llm) # LLM을 사용하여 문서 필터 생성

compression_retriever = ContextualCompressionRetriever(
    base_compressor = _filter,
    base_retriever = retriever
)

compressed_docs = compression_retriever.invoke(
    "Semantic Search에 대해서 알려줘" # 질의
)
pretty_print_docs(compressed_docs) # 검색된 압축된 문서 내용 출력

문서 1:
Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.
예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.
연관키워드: 자연어 처리, 검색 알고리즘, 데이터 마이닝

Embedding


In [37]:
from langchain_classic.retrievers.document_compressors import EmbeddingsFilter
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

embeddings_filter = EmbeddingsFilter(
    embeddings=embeddings, # 임베딩 객체를 전달
    similarity_threshold=0.86, # 유사도 임계값 설정
)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=embeddings_filter,
    base_retriever=retriever
)

compressed_docs = compression_retriever.invoke("Semantic Search에 대해서 알려줘") # 질의
pretty_print_docs(compressed_docs) # 검색된 압축된 문서 내용 출력

문서 1:
Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.
예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.
연관키워드: 자연어 처리, 검색 알고리즘, 데이터 마이닝

Embedding
